# Get to Know a Dataset: Human Cell Atlas

This notebook is a guided tour of the
[Human Cell Atlas](https://registry.opendata.aws/humancellatlas/) (HCA)
dataset, a mirror of all open-access data files in the HCA Data Coordination
Platform. More usage examples, tutorials, and documentation for this dataset
and others can be found at the
[Registry of Open Data on AWS](https://registry.opendata.aws/).

The [Human Cell Atlas](https://www.humancellatlas.org/) is an international
collaboration building comprehensive reference maps of all human cells as a
basis for understanding health and disease. The data mirrored here are
published through the [HCA Data Browser](https://explore.data.humancellatlas.org/),
which is served by [Azul](https://service.azul.data.humancellatlas.org/), an
open-source metadata index and REST web service developed at the UC Santa Cruz
Genomics Institute.

Two things are worth knowing before you start:

1. The S3 bucket is **content-addressed**. Object keys are hashes of file
   content, not file names. You will not find a browsable directory tree of
   `donor/sample/file.fastq.gz` in it.
2. File names, formats, and all other metadata live in the **Azul index**,
   which you query over HTTP. The normal workflow is therefore *query Azul
   first, then fetch from S3*.

This notebook walks through exactly that workflow.

### How is the HCA bucket organized?

The mirror bucket, `s3://humancellatlas` in `us-east-1`, uses content-based
addressing. This lets us mirror efficiently and avoid storing duplicate
content twice: one sequence of bytes is stored exactly once, no matter how
many files in how many projects have that same content.

The layout defines two kinds of objects that you will use, both keyed by a
*digest* &mdash; a hash of the file's content, written as
`${digest_value}.${digest_type}`, where `digest_type` is `sha256`, `sha1` or
`md5`:

| Prefix  | Key                                     | Contents                                    |
| ------- | --------------------------------------- | ------------------------------------------- |
| `file/` | `file/${digest_value}.${digest_type}`      | The file's bytes                            |
| `info/` | `info/${digest_value}.${digest_type}.json` | JSON with the file's `content-type`         |

A third kind, `alias/`, is specified to make a file reachable under hash
algorithms other than the one in its `file/` key. It is not yet populated, so
this notebook sticks to the `sha256` digests that Azul reports. Any other
prefixes you may observe in the bucket are internal to the mirroring process
and are not part of the specification.

The full specification, including the retrieval procedures and the rationale
behind the layout, is at
[docs/mirror.rst](https://github.com/DataBiosphere/azul/blob/develop/docs/mirror.rst)
in the Azul repository.

In [ ]:
# Installs the libraries this notebook needs into the active kernel. Skip
# this cell if your environment already provides them; Google Colab, for one,
# does not provide boto3.
#
# %pip rather than !pip: the magic installs into the kernel's own
# interpreter, whereas pip run in a subshell targets whatever comes first on
# PATH, which is not always the same environment.
#
# The lower bounds are the oldest versions this notebook was tested
# against, on Python 3.9; it was also tested with boto3 1.43, pandas
# 3.0 and matplotlib 3.11 on Python 3.13. The upper bounds stop a
# future major release from breaking it silently.
%pip install -q 'boto3>=1.42,<2' 'pandas>=2.3,<4' 'matplotlib>=3.9,<4'

In [ ]:
import json
import urllib.parse
import urllib.request
import hashlib

import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

In [ ]:
bucket = 'humancellatlas'

# The bucket is public, so requests don't need to be signed. Without this,
# boto3 would look for credentials and fail if it didn't find any.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# List the top-level prefixes
response = s3.list_objects_v2(Bucket=bucket, Delimiter='/')
for item in response['CommonPrefixes']:
    print(item['Prefix'])

Let's look at a few keys under each of the two prefixes we care about. Note
that a `file/` key and its corresponding `info/` key differ only in the prefix
and the `.json` suffix, because both are derived from the same digest.

In [ ]:
for prefix in ['file/', 'info/']:
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=3)
    for item in response['Contents']:
        print(f"{item['Key']}  ({item['Size']:,} bytes)")
    print()

This is the part that surprises most newcomers: the keys tell you nothing
about what the files *are*. That is by design. The same file can be known
under several names in several projects, so names are metadata, and metadata
belongs in the index rather than in the key. It also keeps the layout portable
&mdash; the bucket can be replicated to virtually any file system or object
store without depending on S3-specific features.

So to find something, you query Azul.

### What data formats are available in HCA? What kinds of data are stored using these formats?

The HCA is a collection of hundreds of independently contributed studies, so
the mirror is heterogeneous by nature. Azul's `/index/summary` endpoint gives
a one-call overview of what's in it.

In [ ]:
service = 'https://service.azul.data.humancellatlas.org'


def azul(endpoint, **params):
    """
    Query the Azul REST API and return the parsed JSON response.
    """
    url = f'{service}/index/{endpoint}'
    if params:
        url += '?' + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url) as response:
        return json.load(response)


summary = azul('summary')

print(f"Projects:  {summary['projectCount']:,}")
print(f"Donors:    {summary['donorCount']:,}")
print(f"Specimens: {summary['specimenCount']:,}")
print(f"Files:     {summary['fileCount']:,}")
print(f"Size:      {summary['totalFileSize'] / 1e12:,.1f} TB")

In [ ]:
formats = pd.DataFrame(summary['fileTypeSummaries'])
formats['TB'] = formats['totalSize'] / 1e12
formats = formats[['format', 'count', 'TB']]
formats.sort_values('count', ascending=False).head(12)

The bulk of the bytes is raw sequencing data &mdash; `fastq.gz` and the `bam`
alignments derived from it, with `bai` index files alongside. These are the
standard formats of the field and are best handled with the usual tools
(`samtools`, `pysam`, and so on); at this scale you would normally process
them on EC2 or in a batch system in `us-east-1`, next to the bucket, rather
than download them.

The far smaller tail is where most people start: gene expression matrices as
`csv`, `tsv`, `mtx` (Matrix Market), `loom` and `h5ad`, plus images and
supplementary tables. These are small enough to pull down and open on a
laptop, which is what we'll do next. Note that a format such as `csv.gz` means
exactly what it says &mdash; a gzip-compressed CSV &mdash; and that the
mirror's `info` object reports the *content* type (`text/csv`), not the
compression.

### How do I download and open a data file from HCA?

The procedure has three steps:

1. Ask Azul for the file's metadata, which includes its digest and its
   mirror URI.
2. Fetch the `info/` object to learn the content type.
3. Fetch the `file/` object, which holds the bytes.

We'll use a single-cell gene expression matrix from a study of human
photoreceptors, [Human photoreceptor cells from different macular subregions
have distinct transcriptional
profiles](https://explore.data.humancellatlas.org/projects/8bd2e5f6-9453-4b9b-9c56-59e3a40dc87e).
Here we look it up by name; in practice you would filter by project, organ,
format or any other facet that Azul indexes.

In [ ]:
file_name = 'GSM5175792_donor_5_fovea_counts.csv.gz'

filters = {'fileName': {'is': [file_name]}}
response = azul('files', filters=json.dumps(filters), size=1)
file = response['hits'][0]['files'][0]

print(f"name:   {file['name']}")
print(f"format: {file['format']}  ({', '.join(file['contentDescription'])})")
print(f"size:   {file['size']:,} bytes")
print(f"sha256: {file['sha256']}")
print(f"mirror: {file['azul_mirror_uri']}")

`azul_mirror_uri` is the S3 URI of the `file/` object, of the form
`s3://${bucket}/file/${digest_value}.${digest_type}`. If it is `null`, the
file is not in the mirror &mdash; access it through `azul_url` instead, which
yields a signed URL that also encodes the file's original name and content
type.

We can derive the `info/` key from the same digest.

In [ ]:
# Take the digest straight out of the mirror URI, so that we don't have to
# assume which hash algorithm is the primary one for this file.
key = urllib.parse.urlparse(file['azul_mirror_uri']).path.lstrip('/')
digest = key.removeprefix('file/')

# Step 1: the info object tells us the content type
info = s3.get_object(Bucket=bucket, Key=f'info/{digest}.json')
info = json.load(info['Body'])
print(f"content-type: {', '.join(info['content-type'])}")

# Step 2: the file object holds the bytes
content = s3.get_object(Bucket=bucket, Key=key)['Body'].read()

# The digest in the key is a checksum, so we may as well use it as one
assert hashlib.sha256(content).hexdigest() == file['sha256']
print(f'{len(content):,} bytes downloaded and verified')

# The key carries no name, so save the file under the name Azul reported
with open(file_name, 'wb') as f:
    f.write(content)

The file is a cell-by-gene count matrix. Its first three columns annotate each
cell &mdash; the retinal region it came from, the cell type assigned by the
contributors, and the sequencing library &mdash; and the remaining columns are
raw UMI counts, one per gene.

In [ ]:
matrix = pd.read_csv(file_name, index_col=0)
annotations, counts = matrix.iloc[:, :3], matrix.iloc[:, 3:]

print(f'{counts.shape[0]:,} cells x {counts.shape[1]:,} genes')
print(f'median UMIs per cell:  {counts.sum(axis=1).median():,.0f}')
print(f'median genes per cell: {(counts > 0).sum(axis=1).median():,.0f}')

annotations.head()

### What does HCA data look like?

First, the shape of the whole collection. Azul's summary breaks the estimated
cell count down by organ, which is a fair picture of where the atlas is dense
and where it is still thin.

In [ ]:
organs = pd.DataFrame([
    {'organ': organ, 'cells': entry['totalCellCountByOrgan']}
    for entry in summary['cellCountSummaries']
    # A handful of specimens have no organ recorded
    for organ in entry['organType'] if organ is not None
])
organs = organs.sort_values('cells', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5), dpi=100)
ax.barh(organs['organ'], organs['cells'] / 1e6, color='#3498db')
ax.invert_yaxis()
ax.set_xlabel('Estimated cells (millions)')
ax.set_title('Human Cell Atlas: estimated cells by organ (top 15)')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', linestyle='--', alpha=0.3)
fig.tight_layout()
plt.show()

And now the file we just downloaded. A quick way to gauge whether a count
matrix is behaving is to check that canonical marker genes light up in the
cell types that are supposed to express them. We normalize each cell to counts
per 10,000 UMIs, then average within each annotated cell type.

In [ ]:
markers = {
    'ARR3': 'cones',
    'RHO': 'rods',
    'RLBP1': 'Muller glia',
    'NEFL': 'neurons',
    'CD34': 'endothelium'
}

# Normalize each cell to counts per 10k UMIs, then average per cell type
normalized = counts.div(counts.sum(axis=1), axis=0) * 1e4
common = annotations['celltype'].value_counts()
common = common[common >= 5].index
expression = normalized.loc[annotations['celltype'].isin(common), [*markers]]
expression = expression.groupby(annotations['celltype']).mean()

# Markers differ by an order of magnitude in absolute expression, so scale
# each gene to its own maximum. The printed values are the unscaled means.
# RHO is absent from this sample, so guard against dividing by zero
scaled = expression / expression.max().replace(0, 1)

fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=100)
ax.imshow(scaled, cmap='viridis', aspect='auto', vmin=0, vmax=1)
for y, cell_type in enumerate(expression.index):
    for x, gene in enumerate(markers):
        value = expression.loc[cell_type, gene]
        ax.text(x, y, f'{value:.1f}', ha='center', va='center', fontsize=8,
                color='white' if scaled.loc[cell_type, gene] < 0.5 else 'black')
ax.set_xticks(range(len(markers)))
ax.set_xticklabels([f'{gene}\n({cell_type})'
                    for gene, cell_type in markers.items()], fontsize=9)
ax.set_yticks(range(len(expression)))
ax.set_yticklabels(expression.index)
ax.set_title('Marker expression by annotated cell type, donor 5 fovea\n'
             'colour scaled per gene, labels are mean counts per 10k UMIs',
             fontsize=10)
fig.tight_layout()
plt.show()

Each marker is brightest in exactly the cell type it is supposed to mark:
`ARR3` (cone arrestin) in cones, `RLBP1` in Muller glia, `NEFL` in retinal
ganglion and amacrine cells, `CD34` in endothelium. `RHO` is dark everywhere,
which is the subject of the next section.

### Does the composition of a dissociated retinal sample match the tissue it came from?

It does not, and this matrix shows why that matters. The fovea is the
cone-rich center of the human retina, yet of the 609 cells in this sample only
a dozen are cones and exactly one is a rod. The sample is instead dominated by
Muller glia and retinal ganglion cells. Photoreceptors are large, fragile
cells whose inner and outer segments do not survive dissociation well, so they
are lost before the droplets are ever formed.

This is not a defect in the data; it is a property of the assay, and the same
effect turns up across the atlas wherever fragile cell types are involved. The
practical consequence is that cell-type proportions in a single-cell dataset
are a statement about what survived dissociation, not about tissue
composition. Treat them accordingly, and prefer comparisons in which the
dissociation bias is shared by both sides &mdash; the study this file comes
from contrasts macular *subregions* rather than absolute abundances, which is
the kind of design that keeps the bias from driving the result.

The broader lesson for anyone approaching this mirror: HCA data are contributed
by hundreds of independent labs, each with its own dissociation protocol,
chemistry and annotation vocabulary. Azul's facets let you find the studies
whose protocols are comparable, which is nearly always the first step in any
cross-study analysis.

### How much of the variation between studies of the same organ is biological, and how much is protocol?

The atlas now holds hundreds of projects, many covering the same tissues with
different dissociation protocols, library chemistries and sequencing depths.
Because Azul indexes those protocol choices as queryable facets alongside the
biology, the mirror is an unusually good substrate for quantifying the
technical component directly, rather than regressing it out and hoping.

A tractable way in: pick one well-covered organ, use the `/index/projects`
endpoint to assemble the set of projects covering it, group them by
`libraryConstructionApproach` and `preservationMethod`, and ask how much of the
between-study variance in cell-type proportions those two facets alone explain.
Start with the contributor-supplied matrices &mdash; they are small, they are
already annotated, and they will tell you quickly whether the effect is worth
the cost of reprocessing raw reads uniformly.

That last step is where the mirror earns its keep. The raw `fastq.gz` is all in
one bucket in `us-east-1`, which makes uniform reprocessing a matter of compute
next to the data rather than months of downloads. If you build something on
it, we would like to hear about it &mdash; see the
[HCA Data Portal](https://data.humancellatlas.org/contact).